## **Data Loading & Filtering**

In this initial phase, the objective is to extract a meaningful and manageable subset from the massive original Yelp dataset (approximately 7 million reviews), focusing exclusively on the restaurant domain.

#### 1. Business Filtering Strategy
Filtering occurs in two sequential stages to optimize resources:
* **Category Filter**: Only businesses containing the tag `'Restaurants'` within the `categories` field are retained.
* **Popularity Filter**: To ensure statistical robustness for subsequent analyses, we imposed the following constraint:
    * `MIN_REVIEW_COUNT > 500`: Only "popular" restaurants with more than 500 historical reviews are selected. This mitigates statistical noise arising from businesses with sparse interactions.

#### 2. Loading Optimization
Due to memory constraints preventing the simultaneous loading of millions of reviews, we employ a sequential stream processing approach.
The system iterates through the file line-by-line, rapidly querying a set of valid IDs. Only reviews belonging to the selected restaurants are retained in memory, while all others are instantly discarded to prevent system overload.

#### 3. Data Merging
The final dataset is the result of an **Inner Join** between the reviews surviving the filter and the restaurant metadata (Name, Categories, Average Stars), creating a unified view ready for analysis.

In [ ]:
import tarfile
import json
import os
import pandas as pd
from typing import Set, List, Optional
from google.colab import drive
from tqdm.auto import tqdm

# ==============================================================================
# 1. CONFIGURATION & SETUP
# ==============================================================================
# Paths & Constants
FILE_PATH = '/content/drive/MyDrive/MAGISTRALE/Text_Mining/Yelp JSON/yelp_dataset.tar'
REVIEW_FILENAME_IN_TAR = 'yelp_academic_dataset_review.json'
BUSINESS_FILENAME_IN_TAR = 'yelp_academic_dataset_business.json'

# Filtering Constraints
TARGET_CATEGORY = 'Restaurants'  # Target Business Category
MIN_REVIEW_COUNT = 500           # Minimum number of reviews required per business

# Dataset Metadata (Specific to Yelp Dataset Version)
TOTAL_REVIEWS_EXACT = 6990280
BUSINESS_ESTIMATED = 150346

# Mount Google Drive
if not os.path.exists('/content/drive'):
    try:
        drive.mount('/content/drive', force_remount=True)
        print(" Google Drive mounted successfully.")
    except Exception as e:
        print(f" Error mounting Google Drive: {e}")
        exit()

# ==============================================================================
# 2. DATA LOADING FUNCTIONS
# ==============================================================================

def load_all_business_data(tar_path: str, member_name: str) -> Optional[pd.DataFrame]:
    """
    Extracts and loads the Business dataset from the TAR archive into a DataFrame.

    Args:
        tar_path (str): Path to the .tar file.
        member_name (str): Name of the JSON file inside the archive.

    Returns:
        pd.DataFrame: DataFrame containing business data, or None if error occurs.
    """
    print(f"\n[INFO] Loading Business Data ('{member_name}')...")
    data = []

    try:
        with tarfile.open(tar_path, 'r') as tar:
            member = tar.getmember(member_name)
            with tar.extractfile(member) as f:
                # Iterate through the file with a progress bar
                for line_bytes in tqdm(f, desc="Loading Businesses", unit=" rows", total=BUSINESS_ESTIMATED):
                    line_str = line_bytes.decode('utf-8').strip()
                    if line_str:
                        data.append(json.loads(line_str))

        df = pd.DataFrame(data)
        print(f" Business Data loaded: {len(df)} rows.")
        # Return only essential columns to save memory
        return df[['business_id', 'name', 'categories', 'stars', 'review_count']]

    except Exception as e:
        print(f" Error loading Business Data: {e}")
        return None


def load_filtered_reviews(
    tar_path: str,
    member_name: str,
    business_ids_to_keep: Set[str],
    chunksize: int = 50000
) -> pd.DataFrame:
    """
    Stream-processes the Review JSON file from the TAR archive.
    Filters reviews in real-time based on a set of valid Business IDs.

    Args:
        tar_path (str): Path to the .tar file.
        member_name (str): Name of the JSON file inside the archive.
        business_ids_to_keep (Set[str]): Set of business IDs to retain (O(1) lookup).
        chunksize (int): Number of records to accumulate before creating a temporary DataFrame.

    Returns:
        pd.DataFrame: Filtered DataFrame containing only relevant reviews.
    """
    print(f"\n[INFO] Loading & Filtering Review Data ('{member_name}')...")

    # Optimization Check: Ensure strictly O(1) lookup complexity
    if not isinstance(business_ids_to_keep, set):
        business_ids_to_keep = set(business_ids_to_keep)
        print("   -> Warning: Converted 'business_ids_to_keep' to set for performance optimization.")

    final_reviews_chunks = []
    data_buffer = []
    matched_count = 0

    try:
        with tarfile.open(tar_path, 'r') as tar:
            member = tar.getmember(member_name)
            with tar.extractfile(member) as f:

                # Using specific total for accurate progress tracking
                progress_bar = tqdm(f, desc="Scanning Reviews", total=TOTAL_REVIEWS_EXACT, unit=" reviews")

                for line_bytes in progress_bar:
                    line_str = line_bytes.decode('utf-8').strip()

                    if line_str:
                        obj = json.loads(line_str)

                        # Core Filtering Logic: Keep review only if business_id is in our target set
                        if obj.get('business_id') in business_ids_to_keep:
                            matched_count += 1

                            # Append strictly necessary fields
                            data_buffer.append({
                                'review_id': obj.get('review_id'),
                                'business_id': obj.get('business_id'),
                                'text': obj.get('text'),
                                'stars': obj.get('stars'),
                                'date': obj.get('date')
                            })

                            # Chunk Management: Flush buffer to DataFrame to manage RAM
                            if len(data_buffer) >= chunksize:
                                final_reviews_chunks.append(pd.DataFrame(data_buffer))
                                data_buffer = [] # Reset buffer

                # Flush remaining buffer after loop
                if data_buffer:
                    final_reviews_chunks.append(pd.DataFrame(data_buffer))

        if final_reviews_chunks:
            df_filtered = pd.concat(final_reviews_chunks, ignore_index=True)
            print(f" filtering Complete. Total matched reviews: {len(df_filtered)}")
            return df_filtered
        else:
            print(" No reviews matched the filtering criteria.")
            return pd.DataFrame()

    except Exception as e:
        print(f" Error during filtered review loading: {e}")
        return pd.DataFrame()

# ==============================================================================
# 3. EXECUTION PIPELINE
# ==============================================================================

# STEP 1: Load all Business Data
df_business_all = load_all_business_data(FILE_PATH, BUSINESS_FILENAME_IN_TAR)

if df_business_all is not None:

    # STEP 2: Filter Business Data
    # 2a. Category Filter
    mask_category = df_business_all['categories'].str.contains(TARGET_CATEGORY, case=False, na=False)
    # 2b. Popularity Filter (Review Count)
    mask_popularity = df_business_all['review_count'] > MIN_REVIEW_COUNT

    # Apply Filters -> Renamed to 'df_business_restaurants' for consistency with Chunk 2
    df_business_restaurants = df_business_all[mask_category & mask_popularity].copy()
    df_business_restaurants = df_business_restaurants.sort_values(by='review_count', ascending=False)

    # Create Optimized Lookup Set
    target_business_ids = set(df_business_restaurants['business_id'].unique())

    print("\n" + "="*60)
    print(f" BUSINESS FILTER SUMMARY")
    print(f"   Category: '{TARGET_CATEGORY}' | Min Reviews: {MIN_REVIEW_COUNT}")
    print(f"   Selected Businesses: {len(df_business_restaurants)}")
    print("="*60)

    # STEP 3: Load Filtered Reviews
    # Renamed to 'df_review_filtered' for consistency with Chunk 2
    df_review_filtered = load_filtered_reviews(
        FILE_PATH,
        REVIEW_FILENAME_IN_TAR,
        target_business_ids
    )

    # STEP 4: Merge and Finalize
    if not df_review_filtered.empty:
        print("\n[INFO] Merging Datasets...")

        # Renamed to 'df_joined' for consistency with Chunk 2
        df_joined = pd.merge(
            df_review_filtered,
            df_business_restaurants[['business_id', 'name', 'categories', 'stars', 'review_count']],
            on='business_id',
            how='inner',
            suffixes=('_review', '_business')
        )

        print("\n" + "="*60)
        print(" DATASET CREATION SUCCESSFUL")
        print("="*60)
        print("Final DataFrame Structure:")
        df_joined.info(memory_usage='deep')

        # Optional: Preview
        print("\nSample Rows:")
        display(df_joined.head(3))

    else:
        print(" Process stopped: No reviews found.")
else:
    print(" Process stopped: Business data failed to load.")

In [ ]:
import os
import pandas as pd

# ==============================================================================
# 4. DATA EXPORT & PERSISTENCE
# ==============================================================================

# Directory Configuration
SAVE_DIR = '/content/drive/MyDrive/MAGISTRALE/Text_Mining/Datasets'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"\n[INFO] Starting Data Export to: {SAVE_DIR}...")

# ------------------------------------------------------------------------------
# 4.1 Save Final Merged Dataset
# ------------------------------------------------------------------------------
# Check if variable exists to avoid NameError if previous steps failed
if 'df_joined' in locals() and df_joined is not None:
    final_file_path = os.path.join(SAVE_DIR, 'df_joined.csv')
    df_joined.to_csv(final_file_path, index=False)
    print(f" Final Dataset saved: {final_file_path}")
    print(f"   Rows: {len(df_joined)}")
else:
    print(" Warning: 'df_joined' not found. Skipping final dataset save.")

# ------------------------------------------------------------------------------
# 4.2 Save Intermediate Business Data (Filtered)
# ------------------------------------------------------------------------------
if 'df_business_restaurants' in locals() and df_business_restaurants is not None:
    business_file_path = os.path.join(SAVE_DIR, 'df_business_restaurants.csv')
    df_business_restaurants.to_csv(business_file_path, index=False)
    print(f" Business Data (Filtered) saved: {business_file_path}")
    print(f"   Rows: {len(df_business_restaurants)}")
else:
    print(" Warning: 'df_business_restaurants' not found. Skipping save.")

# ------------------------------------------------------------------------------
# 4.3 Save Intermediate Review Data (Filtered)
# ------------------------------------------------------------------------------
if 'df_review_filtered' in locals() and df_review_filtered is not None:
    review_file_path = os.path.join(SAVE_DIR, 'df_review_restaurants.csv')
    df_review_filtered.to_csv(review_file_path, index=False)
    print(f" Review Data (Filtered) saved: {review_file_path}")
    print(f"   Rows: {len(df_review_filtered)}")
else:
    print(" Warning: 'df_review_filtered' not found. Skipping save.")


print("\n" + "="*50)
print(" EXPORT PROCESS COMPLETED SUCCESSFULLY")
print("="*50)

In [ ]:
df_joined